# Amazon Electronics: поиск товаров с худшими отзывами

Агрегация отзывов Amazon (категория Electronics) на PySpark: средние рейтинги, доля плохо оценённых товаров и топ-10 худших с названиями из метаданных.

In [ ]:
# Зависимости: Java 11+ и PySpark
%pip install -q pyspark==3.5.1

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("ElectronicsDataAnalysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Чтение meta-данных как JSON
meta_df = spark.read.json("data/meta_Electronics.json")
meta_df = meta_df.filter(col("asin").isNotNull()).select("asin", "title")

# Чтение отзывов
reviews_df = spark.read.json("data/Electronics_5.json")

# Средний рейтинг по каждому товару
avg_rating_df = reviews_df.groupBy("asin") \
    .agg(avg("overall").alias("avg_rating")) \
    .filter(col("avg_rating").isNotNull())

# Общий средний рейтинг по всем отзывам
total_avg_review = reviews_df.select(avg("overall")).collect()[0][0]
print(f"Средний рейтинг по всем отзывам: {total_avg_review:.2f}")

# Среднее значение средних рейтингов товаров
avg_of_avg_rating = avg_rating_df.select(avg("avg_rating")).collect()[0][0]
print(f"Среднее значение средних рейтингов товаров: {avg_of_avg_rating:.2f}")

product_rating_df = avg_rating_df.join(meta_df, on="asin", how="inner")

# Товары с рейтингом меньше 3
low_rated_df = product_rating_df.filter(col("avg_rating") < 3.0)
low_count = low_rated_df.count()
total_products = avg_rating_df.count()
print(f"Товаров с рейтингом меньше 3: {low_count} из {total_products} ({100 * low_count / total_products:.2f}%)")

top10_worst = low_rated_df.orderBy("avg_rating").limit(10)
print("10 товаров с рейтингом меньше 3:")
top10_worst.show(truncate=False)

output_dir = "output/worst_rated_electronics"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

result_df = top10_worst.select(
    concat_ws("\t", format_number(col("avg_rating"), 4), col("title")).alias("result")
)
result_df.write.mode("overwrite").text(output_dir)

print("Результат сохранен в папку:", output_dir)

Средний рейтинг по всем отзывам: 4.25
Среднее значение средних рейтингов товаров: 4.15
Товаров с рейтингом меньше 3: 140 из 2866 (4.88%)
10 товаров с рейтингом меньше 3:
+----------+------------------+--------------------------------------------------------------------------------+
|asin      |avg_rating        |title                                                                           |
+----------+------------------+--------------------------------------------------------------------------------+
|B00000JBIA|1.0               |Agfa ePhoto SMILE 0.2MP Digital Camera                                          |
|B00005T3Y9|1.1428571428571428|Terk AF-9330 AM/FM Indoor Amplified Tower Antenna                               |
|B00005T3XR|1.2857142857142858|Jensen DSC-IR100A Remote Control Extender (Discontinued by Manufacturer)        |
|B00007GWC8|1.8               |Digital Concepts TR-036CL Mini Tripod with Extension Legs and Tilt Head (Silver)|
|B00004U8K4|1.8               |Plantron

## Выводы

- Средний рейтинг по всем отзывам **4.25**, среднее средних рейтингов товаров **4.15**, то есть большинство товаров оценивают положительно.
- Из 2866 товаров с отзывами у **140 (4.88%)** средний рейтинг ниже 3.0.
- В топ-10 худших попали камеры, аксессуары и устройства, снятые с производства. Низкие оценки чаще получает устаревшая техника и малоизвестные бренды.